In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import scipy.stats
import matplotlib.pyplot as plt
import re, json
from sklearn.metrics import roc_auc_score
import os
import xgboost as xgb  # Import XGBoost

# Import the original feature calculation utility
import utils

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

params = json.load(open("siRNA_param.json", "r"))

score_PCC = []
score_SPCC = []
score_mse = []
score_auc = []

for n in range(3):
    print(f"--- Processing Split {n} ---")

    """
    1. Read File
    """
    data_train = pd.read_csv("siRNA_split_datasets/split" + str(n) + "/train.csv")
    data_dev = pd.read_csv("siRNA_split_datasets/split" + str(n) + "/dev.csv")
    data_test = pd.read_csv("siRNA_split_datasets/split" + str(n) + "/test.csv")

    data_train["siRNA_seq"] = data_train["siRNA_seq"].replace("U", "T", regex=True)
    data_dev["siRNA_seq"] = data_dev["siRNA_seq"].replace("U", "T", regex=True)
    data_test["siRNA_seq"] = data_test["siRNA_seq"].replace("U", "T", regex=True)

    data = pd.concat([data_train, data_dev, data_test], axis=0).reset_index(drop=True)

    """
    2. Feature Processing (Corrected and Improved)
    """
    # one-hot
    ## siRNA
    sirna_onehot = [
        utils.obtain_one_hot_feature_for_one_sequence_1(seq, params["sirna_length"])
        for seq in data["siRNA_seq"]
    ]
    sirna_onehot = pd.DataFrame(
        sirna_onehot, index=list(data["siRNA"])
    ).drop_duplicates()

    ## mRNA (use de-duplicated temp dataframe for all mRNA feature calculations)
    mrna_onehot_temp = (
        data.loc[:, ["mRNA", "mRNA_seq"]].drop_duplicates(subset="mRNA").copy()
    )
    # FIX: Clean the mRNA sequences to remove non-standard characters like 'N'
    mrna_onehot_temp["mRNA_seq"] = mrna_onehot_temp["mRNA_seq"].apply(
        lambda seq: re.sub(r"[^ATCG]", "", seq)
    )

    mrna_onehot = [
        utils.obtain_one_hot_feature_for_one_sequence_1(seq, params["max_mrna_len"])
        for seq in mrna_onehot_temp["mRNA_seq"]
    ]
    mrna_onehot = pd.DataFrame(mrna_onehot, index=list(mrna_onehot_temp["mRNA"]))

    # Positional encoding
    trans_table = str.maketrans("ATCG", "TAGC")
    data["match_pos"] = [
        seq[::-1].upper().translate(trans_table) for seq in data["siRNA_seq"]
    ]
    # FIX: Use .find() for safety; it won't raise an error if not found.
    data["match_pos"] = data.apply(
        lambda row: row["mRNA_seq"].find(row["match_pos"]), axis=1
    )
    sirna_pos_encoding = [
        utils.get_pos_embedding_sequence(num, params["sirna_length"], params["dmodel"])
        for num in data["match_pos"]
    ]
    # FIX: Correctly set the index to be the interaction index
    sirna_pos_encoding = pd.DataFrame(
        sirna_pos_encoding, index=data["siRNA"] + "_" + data["mRNA"]
    )

    # Thermodynamics
    sirna_thermo_feat = [
        utils.cal_thermo_feature(seq.replace("T", "U")) for seq in data["siRNA_seq"]
    ]
    sirna_thermo_feat = pd.DataFrame(sirna_thermo_feat)
    sirna_thermo_feat.index = data["siRNA"] + "_" + data["mRNA"]

    # Co-fold features
    con_feat = pd.read_csv(
        "siRNA_split_preprocess/con_matrix.txt", header=None, index_col=0
    )
    con_feat = con_feat.reindex(sirna_thermo_feat.index)

    # sel-fold features
    sirna_sfold_feat = pd.read_csv(
        "siRNA_split_preprocess/self_siRNA_matrix.txt", header=None, index_col=0
    ).reindex(sirna_onehot.index)
    mrna_sfold_feat = pd.read_csv(
        "siRNA_split_preprocess/self_mRNA_matrix.txt", header=None, index_col=0
    ).reindex(mrna_onehot.index)

    # AGO2
    sirna_ago = pd.read_csv("RNA_AGO2/siRNA_AGO2.csv", index_col=0).reindex(
        sirna_onehot.index
    )
    mrna_ago = pd.read_csv("RNA_AGO2/mRNA_AGO2.csv", index_col=0).reindex(
        mrna_onehot.index
    )

    # GC percentage
    sirna_GC = pd.DataFrame(
        [utils.countGC(seq) for seq in data["siRNA_seq"]], index=list(data["siRNA"])
    ).drop_duplicates()
    mrna_GC = pd.DataFrame(
        [utils.countGC(seq) for seq in mrna_onehot_temp["mRNA_seq"]],
        index=list(mrna_onehot_temp["mRNA"]),
    )

    # k-mers
    sirna_1_mer = pd.DataFrame([utils.single_freq(seq) for seq in data["siRNA_seq"]])
    sirna_2_mers = pd.DataFrame([utils.double_freq(seq) for seq in data["siRNA_seq"]])
    sirna_3_mers = pd.DataFrame([utils.triple_freq(seq) for seq in data["siRNA_seq"]])
    sirna_k_mers = pd.concat([sirna_1_mer, sirna_2_mers, sirna_3_mers], axis=1)
    sirna_k_mers.index = data["siRNA"]
    sirna_k_mers = sirna_k_mers.loc[~sirna_k_mers.index.duplicated(keep="first")]

    # siRNA rules codes
    sirna_pos_scores = pd.DataFrame(
        [utils.rules_scores(seq) for seq in data["siRNA_seq"]],
        index=list(data["siRNA"]),
    ).drop_duplicates()

    # FIX: Add unique prefixes to all feature columns to prevent duplicate column names
    sirna_onehot.columns = "soh_" + sirna_onehot.columns.astype(str)
    sirna_sfold_feat.columns = "ssf_" + sirna_sfold_feat.columns.astype(str)
    sirna_ago.columns = "sago_" + sirna_ago.columns.astype(str)
    sirna_GC.columns = "sgc_" + sirna_GC.columns.astype(str)
    sirna_k_mers.columns = "skm_" + sirna_k_mers.columns.astype(str)
    sirna_pos_scores.columns = "sps_" + sirna_pos_scores.columns.astype(str)

    mrna_onehot.columns = "moh_" + mrna_onehot.columns.astype(str)
    mrna_sfold_feat.columns = "msf_" + mrna_sfold_feat.columns.astype(str)
    mrna_ago.columns = "mago_" + mrna_ago.columns.astype(str)
    mrna_GC.columns = "mgc_" + mrna_GC.columns.astype(str)

    sirna_thermo_feat.columns = "itf_" + sirna_thermo_feat.columns.astype(str)
    con_feat.columns = "icf_" + con_feat.columns.astype(str)
    sirna_pos_encoding.columns = "ipe_" + sirna_pos_encoding.columns.astype(str)

    """
    3. Data Restructuring for XGBoost: Create a "flattened" feature table
    """
    # Combine all siRNA and mRNA features into their own dataframes
    sirna_pd = pd.concat(
        [
            sirna_onehot,
            sirna_sfold_feat,
            sirna_ago,
            sirna_GC,
            sirna_k_mers,
            sirna_pos_scores,
        ],
        axis=1,
    )
    mrna_pd = pd.concat([mrna_onehot, mrna_sfold_feat, mrna_ago, mrna_GC], axis=1)
    interaction_pd = pd.concat(
        [sirna_thermo_feat, con_feat, sirna_pos_encoding], axis=1
    )

    # Merge all features into a single dataframe based on the main 'data' table
    flat_data = pd.merge(data, sirna_pd, left_on="siRNA", right_index=True, how="left")
    flat_data = pd.merge(
        flat_data, mrna_pd, left_on="mRNA", right_index=True, how="left"
    )
    flat_data["interaction_index"] = flat_data["siRNA"] + "_" + flat_data["mRNA"]
    flat_data = pd.merge(
        flat_data,
        interaction_pd,
        left_on="interaction_index",
        right_index=True,
        how="left",
    )

    # Define the target variable (y)
    y = flat_data["efficacy"]

    # Define all columns that are NOT features and should be dropped
    cols_to_drop = [
        "siRNA",
        "mRNA",
        "siRNA_seq",
        "mRNA_seq",
        "efficacy",
        "match_pos",
        "interaction_index",
    ]

    # Create the initial feature matrix (X) by dropping the non-feature columns
    X = flat_data.drop(columns=cols_to_drop)

    # FIX: Ensure X contains ONLY numeric columns to prevent dtype errors
    X = X.select_dtypes(include=np.number)

    print(
        f"Created feature matrix X with {X.shape[0]} samples and {X.shape[1]} numeric features."
    )

    # Split the cleaned data back into train, dev, and test sets using original row counts
    train_size = len(data_train)
    dev_size = len(data_dev)

    X_train, y_train = X.iloc[:train_size], y.iloc[:train_size]
    X_dev, y_dev = (
        X.iloc[train_size : train_size + dev_size],
        y.iloc[train_size : train_size + dev_size],
    )
    X_test, y_test = X.iloc[train_size + dev_size :], y.iloc[train_size + dev_size :]

    """
    4. Model Training with XGBoost
    """
    # Initialize the XGBoost Regressor model
    xgb_model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=1000,  # Max number of trees to build
        learning_rate=0.05,
        early_stopping_rounds=50,  # Stop if validation loss doesn't improve for 50 rounds
        n_jobs=-1,  # Use all available CPU cores
        random_state=42,
    )

    print("Training XGBoost model...")
    # Train the model, using the development set for early stopping
    xgb_model.fit(X_train, y_train, eval_set=[(X_dev, y_dev)], verbose=False)

    """
    5. Evaluate on the Test Set
    """
    print("Evaluating on the test set...")
    # Predict on the test set
    test = xgb_model.predict(X_test)

    """
    6. Calculate Metrics
    """
    # PCC
    pearson = scipy.stats.pearsonr(y_test, test)
    score_PCC.append(pearson[0])
    print(f"PCC: {pearson[0]}")

    # SPCC
    spearman = scipy.stats.spearmanr(y_test, test)
    score_SPCC.append(spearman[0])
    print(f"SPCC: {spearman[0]}")

    # MSE
    mse_run = mean_squared_error(y_test, test)
    score_mse.append(mse_run)
    print(f"MSE: {mse_run}")

    # AUC
    binary_pred = (y_test > 0.7).astype(int)
    auc = roc_auc_score(binary_pred, test)
    score_auc.append(auc)
    print(f"AUC: {auc}")

    print(f"Split {n} finished!\n")


# Print the average value of metrics across all splits
print("--- Overall Results ---")
print(f"Overall PCC score = {np.mean(score_PCC)}")
print(f"Overall SPCC score = {np.mean(score_SPCC)}")
print(f"Overall MSE score = {np.mean(score_mse)}")
print(f"Overall AUC score = {np.mean(score_auc)}")

ModuleNotFoundError: No module named 'pandas'

In [2]:
!pip install biopython
!pip install tqdm seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.7 MB/s  0:00:00m 10.1 MB/s eta 0:00:01
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [seaborn]━━━ 1/2 [seaborn]
